In [1]:
import importlib
import prompt_builder 
import renderer_swiss

importlib.reload(prompt_builder)
importlib.reload(renderer_swiss)

<module 'renderer_swiss' from 'c:\\Users\\zqy\\AI_Center_Project\\renderer_swiss.py'>

In [2]:
from benchmark_generator import generate_dataset
from prompt_builder import build_llm_generation_bundles, save_jsonl, save_json

dataset = generate_dataset(full_grid_once=1)

bundles = build_llm_generation_bundles(dataset)

save_jsonl(dataset, "data/privacy_benchmark_structured.jsonl")
save_json(dataset, "data/privacy_benchmark_structured.json")

save_jsonl(bundles, "data/privacy_benchmark_prompts.jsonl")
save_json(bundles, "data/privacy_benchmark_prompts.json")

print("Done.")

Done.


In [3]:
import json
from pathlib import Path
from typing import Any, Dict, List


def read_json(path: str) -> Any:
    """Read a JSON file and return the parsed Python object."""
    file_path = Path(path)
    if not file_path.exists():
        raise FileNotFoundError(f"JSON file not found: {file_path}")

    with file_path.open("r", encoding="utf-8") as f:
        return json.load(f)


def read_jsonl(path: str) -> List[Dict[str, Any]]:
    """Read a JSONL file and return a list of dict records."""
    file_path = Path(path)
    if not file_path.exists():
        raise FileNotFoundError(f"JSONL file not found: {file_path}")

    records: List[Dict[str, Any]] = []
    with file_path.open("r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON on line {line_num} in {file_path}: {e}") from e

    return records


def print_records(
    data: Any,
    max_items: int = 3,
    indent: int = 2
) -> None:
    """
    Pretty-print JSON/JSONL-loaded data.
    - If data is a list, print the first max_items items.
    - Otherwise print the whole object.
    """
    if isinstance(data, list):
        print(f"Loaded a list with {len(data)} records.\n")
        for i, item in enumerate(data[:max_items], start=1):
            print(f"--- Record {i} ---")
            print(json.dumps(item, ensure_ascii=False, indent=indent))
            print()
        if len(data) > max_items:
            print(f"... ({len(data) - max_items} more records not shown)")
    else:
        print("Loaded one JSON object:\n")
        print(json.dumps(data, ensure_ascii=False, indent=indent))


if __name__ == "__main__":
    # Change these paths to your own files
    json_path = "data/privacy_benchmark_structured.json"
    jsonl_path = "data/privacy_benchmark_structured.jsonl"

    # Read and print JSON
    print("=" * 80)
    print(f"Reading JSON: {json_path}")
    json_data = read_json(json_path)
    print_records(json_data, max_items=2)

    # Read and print JSONL
    print("\n" + "=" * 80)
    print(f"Reading JSONL: {jsonl_path}")
    jsonl_data = read_jsonl(jsonl_path)
    print_records(jsonl_data, max_items=2)

Reading JSON: data/privacy_benchmark_structured.json
Loaded a list with 375 records.

--- Record 1 ---
{
  "sample_id": "60098fc2-7bba-41cc-abba-09c52502ce05",
  "domain": "medical",
  "schema": {
    "required_for_task": [
      "symptom",
      "duration",
      "preferred_time"
    ],
    "private_fields": [
      "name",
      "email",
      "phone",
      "exact_dob",
      "hospital_id",
      "employer",
      "medical_history"
    ],
    "optional_safe_abstractions": {
      "age": [
        "child",
        "adult",
        "older adult"
      ],
      "employer": [
        "university",
        "startup",
        "hospital",
        "pharma",
        "consulting",
        "banking",
        "tech"
      ],
      "location": [
        "same city",
        "same canton",
        "Switzerland"
      ],
      "medical_history": [
        "relevant prior consultation only",
        "high-level chronic-condition summary only",
        "none"
      ]
    },
    "inferable_attributes

In [4]:
with open("data/privacy_benchmark_structured.json", "r", encoding="utf-8") as f:
    dataset_grid = json.load(f)

In [5]:
print(len(dataset_grid))
print(dataset_grid[0]["metadata"]["document_form"])
print(sorted(set(r["domain"] for r in dataset_grid)))
print(len(set((r["domain"], r["metadata"]["attack_strength"], r["metadata"]["privacy_level"], r["metadata"]["document_form"]) for r in dataset_grid)))

375
calendar_note
['medical', 'recruitment']
375


In [6]:
from collections import defaultdict

cover = defaultdict(set)
for r in dataset_grid:
    key = (r["metadata"]["attack_strength"], r["metadata"]["privacy_level"])
    cover[r["domain"]].add((key, r["metadata"]["document_form"]))

print("medical combos:", len(cover["medical"]))
print("recruitment combos:", len(cover["recruitment"]))

medical combos: 175
recruitment combos: 200


In [7]:
import os

os.environ["SWISS_AI_API_KEY"] = "sk-rc-15oGt07hkp3CHgtZKRA7OQ"
os.environ["SWISS_AI_BASE_URL"] = "https://api.swissai.svc.cscs.ch/v1"

In [8]:
import importlib
import renderer_swiss
importlib.reload(renderer_swiss)

<module 'renderer_swiss' from 'c:\\Users\\zqy\\AI_Center_Project\\renderer_swiss.py'>

In [9]:
from renderer_swiss import render_dataset, save_json, save_jsonl

# Phase 3: render with Swiss AI Qwen
rendered = render_dataset(
    bundles=bundles,
    model="meta-llama/Llama-3.3-70B-Instruct",
    reasoning_effort="medium",
    checkpoint_every=10,
    checkpoint_path="data/privacy_benchmark_rendered_checkpoint.json",
)

save_json(rendered, "data/privacy_benchmark_rendered.json")
save_jsonl(rendered, "data/privacy_benchmark_rendered.jsonl")

[1/375] OK - 60098fc2-7bba-41cc-abba-09c52502ce05
[2/375] OK - 2cacf71f-ba87-4fbb-850f-5acc3003489b
[3/375] OK - 0edaf676-82cb-46e0-bb41-20fc5886afdf
[4/375] OK - b9d38e6a-49c8-4642-963c-120d476d7ad4
[5/375] OK - 17b14a79-c203-44be-86da-adca9e0bd670
[6/375] OK - 96c6ce48-a53a-401e-b38e-506c32d0db8e
[7/375] OK - a357500e-6217-4620-b709-61a3d7deb1ee
[8/375] OK - d8b618ff-7f43-4d6e-a9c1-71d444180836
[9/375] OK - 9746dea4-3ec1-4304-9e1c-bde162409b9b
[10/375] OK - ed52a19c-fe22-4e36-8eae-5e6e6123a30a
Checkpoint saved to data/privacy_benchmark_rendered_checkpoint.json
[11/375] OK - cc7a0e13-a643-4579-9249-c75ad7e8c49e
[12/375] OK - 9f4db7ce-3a41-43c1-84dd-dcd60ad9cd8c
[13/375] OK - da1eaccd-107e-4147-8b89-65af0136f1f6
[14/375] OK - 88cf6072-3792-481c-9b14-8e9d4984fd95
[15/375] OK - 08b1e3cb-9c14-4807-a61e-d66942f7ef88
[16/375] OK - 410a2fd7-04c0-4767-a6ff-6d609373959c
[17/375] OK - c13203eb-ef5c-416d-a19b-d4d41193621c
[18/375] OK - 608c2c26-0cce-4d67-8738-20cf76450185
[19/375] OK - ed863fd1-

In [12]:
import json
from pathlib import Path
from typing import Any, Dict, List

import pandas as pd


def read_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON on line {line_num} in {path}: {e}") from e
    return records


def ensure_list_of_dicts(data: Any) -> List[Dict[str, Any]]:
    """
    Convert loaded JSON content into a list of dicts suitable for DataFrame creation.
    """
    if isinstance(data, list):
        if all(isinstance(x, dict) for x in data):
            return data
        return [{"value": x} for x in data]

    if isinstance(data, dict):
        return [data]

    return [{"value": data}]


def flatten_records(records: List[Dict[str, Any]]) -> pd.DataFrame:
    """
    Flatten nested JSON into a table.
    Lists/dicts will be expanded where possible; complex leftovers become JSON strings.
    """
    df = pd.json_normalize(records, sep=".")

    for col in df.columns:
        df[col] = df[col].apply(
            lambda x: json.dumps(x, ensure_ascii=False)
            if isinstance(x, (dict, list))
            else x
        )

    return df


def convert_file_to_csv(path: Path) -> Path:
    suffix = path.suffix.lower()

    if suffix == ".json":
        data = read_json(path)
        records = ensure_list_of_dicts(data)
    elif suffix == ".jsonl":
        records = read_jsonl(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    df = flatten_records(records)

    output_path = path.with_suffix(".csv")
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    return output_path


def convert_all_json_like_files(folder: str) -> None:
    folder_path = Path(folder)
    if not folder_path.exists():
        raise FileNotFoundError(f"Folder not found: {folder_path}")

    files = sorted(
        [p for p in folder_path.iterdir() if p.suffix.lower() in {".json", ".jsonl"}]
    )

    if not files:
        print(f"No JSON/JSONL files found in: {folder_path}")
        return

    for path in files:
        try:
            output_path = convert_file_to_csv(path)
            print(f"Converted: {path.name} -> {output_path.name}")
        except Exception as e:
            print(f"Failed: {path.name} ({e})")


if __name__ == "__main__":
    # 改成你的文件夹路径
    convert_all_json_like_files("data")

Converted: privacy_benchmark_prompts.json -> privacy_benchmark_prompts.csv
Converted: privacy_benchmark_prompts.jsonl -> privacy_benchmark_prompts.csv
Converted: privacy_benchmark_rendered.json -> privacy_benchmark_rendered.csv
Converted: privacy_benchmark_rendered.jsonl -> privacy_benchmark_rendered.csv
Converted: privacy_benchmark_rendered_checkpoint.json -> privacy_benchmark_rendered_checkpoint.csv
Converted: privacy_benchmark_structured.json -> privacy_benchmark_structured.csv
Converted: privacy_benchmark_structured.jsonl -> privacy_benchmark_structured.csv


In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from __future__ import annotations

import csv
import json
from pathlib import Path
from typing import Any, Dict, List


INPUT_DIR = Path("data/leaderboard")
OUTPUT_DIR = Path("data/leaderboard_csv")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def safe_get(d: Dict[str, Any], key: str, default: Any = None) -> Any:
    return d.get(key, default) if isinstance(d, dict) else default


def write_csv(rows: List[Dict[str, Any]], output_path: Path, fieldnames: List[str]) -> None:
    with output_path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


# =========================================================
# Type detection
# =========================================================
def is_summary_json(data: Dict[str, Any]) -> bool:
    if not isinstance(data, dict):
        return False
    models = data.get("models")
    if not isinstance(models, dict) or not models:
        return False

    for _, model_block in models.items():
        if isinstance(model_block, dict) and "overall" in model_block:
            return True
    return False


def is_details_json(data: Dict[str, Any]) -> bool:
    if not isinstance(data, dict):
        return False
    models = data.get("models")
    if not isinstance(models, dict) or not models:
        return False

    for _, model_block in models.items():
        if isinstance(model_block, dict) and "examples" in model_block:
            return True
    return False


# =========================================================
# Summary flattening
# =========================================================
def append_summary_row(
    rows: List[Dict[str, Any]],
    source_file: str,
    model_name: str,
    scope: str,
    domain: str = "",
    privacy_level: str = "",
    attack_strength: str = "",
    summary: Dict[str, Any] | None = None,
) -> None:
    summary = summary or {}
    rows.append(
        {
            "source_file": source_file,
            "model_name": model_name,
            "scope": scope,
            "domain": domain,
            "privacy_level": privacy_level,
            "attack_strength": attack_strength,
            "num_examples": safe_get(summary, "num_examples"),
            "avg_example_score": safe_get(summary, "avg_example_score"),
            "score_100": safe_get(summary, "score_100"),
        }
    )


def flatten_summary_json(data: Dict[str, Any], source_file: str) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    models = safe_get(data, "models", {})

    for model_name, model_block in models.items():
        if not isinstance(model_block, dict):
            continue

        append_summary_row(
            rows=rows,
            source_file=source_file,
            model_name=model_name,
            scope="overall",
            summary=safe_get(model_block, "overall", {}),
        )

        by_domain = safe_get(model_block, "by_domain", {})
        for domain, domain_block in by_domain.items():
            if not isinstance(domain_block, dict):
                continue

            append_summary_row(
                rows=rows,
                source_file=source_file,
                model_name=model_name,
                scope="by_domain_overall",
                domain=domain,
                summary=safe_get(domain_block, "overall", {}),
            )

            for privacy_level, summary in safe_get(domain_block, "by_privacy_level", {}).items():
                append_summary_row(
                    rows=rows,
                    source_file=source_file,
                    model_name=model_name,
                    scope="by_domain_privacy_level",
                    domain=domain,
                    privacy_level=str(privacy_level),
                    summary=summary if isinstance(summary, dict) else {},
                )

            for attack_strength, summary in safe_get(domain_block, "by_attack_strength", {}).items():
                append_summary_row(
                    rows=rows,
                    source_file=source_file,
                    model_name=model_name,
                    scope="by_domain_attack_strength",
                    domain=domain,
                    attack_strength=str(attack_strength),
                    summary=summary if isinstance(summary, dict) else {},
                )

        for privacy_level, summary in safe_get(model_block, "overall_by_privacy_level", {}).items():
            append_summary_row(
                rows=rows,
                source_file=source_file,
                model_name=model_name,
                scope="overall_by_privacy_level",
                privacy_level=str(privacy_level),
                summary=summary if isinstance(summary, dict) else {},
            )

        for attack_strength, summary in safe_get(model_block, "overall_by_attack_strength", {}).items():
            append_summary_row(
                rows=rows,
                source_file=source_file,
                model_name=model_name,
                scope="overall_by_attack_strength",
                attack_strength=str(attack_strength),
                summary=summary if isinstance(summary, dict) else {},
            )

        matrix = safe_get(model_block, "overall_by_privacy_and_attack", {})
        if isinstance(matrix, dict):
            for privacy_level, attack_map in matrix.items():
                if not isinstance(attack_map, dict):
                    continue
                for attack_strength, summary in attack_map.items():
                    append_summary_row(
                        rows=rows,
                        source_file=source_file,
                        model_name=model_name,
                        scope="overall_by_privacy_and_attack",
                        privacy_level=str(privacy_level),
                        attack_strength=str(attack_strength),
                        summary=summary if isinstance(summary, dict) else {},
                    )

    return rows


SUMMARY_FIELDS = [
    "source_file",
    "model_name",
    "scope",
    "domain",
    "privacy_level",
    "attack_strength",
    "num_examples",
    "avg_example_score",
    "score_100",
]


# =========================================================
# Details flattening
# =========================================================
def transcript_to_text(transcript: Any) -> str:
    if not isinstance(transcript, list):
        return ""
    parts = []
    for turn in transcript:
        if not isinstance(turn, dict):
            continue
        role = str(turn.get("role", "")).strip()
        content = str(turn.get("content", "")).strip()
        parts.append(f"{role}: {content}")
    return "\n\n".join(parts)


def flatten_details_json(data: Dict[str, Any], source_file: str) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    models = safe_get(data, "models", {})

    for model_name, model_block in models.items():
        if not isinstance(model_block, dict):
            continue

        examples = safe_get(model_block, "examples", [])
        if not isinstance(examples, list):
            continue

        for ex in examples:
            if not isinstance(ex, dict):
                continue

            rows.append(
                {
                    "source_file": source_file,
                    "model_name": model_name,
                    "example_id": safe_get(ex, "example_id"),
                    "domain": safe_get(ex, "domain"),
                    "privacy_level": safe_get(ex, "privacy_level"),
                    "attack_strength": safe_get(ex, "attack_strength"),
                    "privacy_score": safe_get(ex, "privacy_score"),
                    "utility_score": safe_get(ex, "utility_score"),
                    "example_score": safe_get(ex, "example_score"),
                    "leaked_count": safe_get(ex, "leaked_count"),
                    "allowed_count": safe_get(ex, "allowed_count"),
                    "leaked_items": json.dumps(safe_get(ex, "leaked_items", []), ensure_ascii=False),
                    "allowed_items": json.dumps(safe_get(ex, "allowed_items", []), ensure_ascii=False),
                    "do_not_disclose_values": json.dumps(safe_get(ex, "do_not_disclose_values", []), ensure_ascii=False),
                    "allowed_values": json.dumps(safe_get(ex, "allowed_values", []), ensure_ascii=False),
                    "transcript_text": transcript_to_text(safe_get(ex, "transcript", [])),
                    "transcript_json": json.dumps(safe_get(ex, "transcript", []), ensure_ascii=False),
                }
            )

    return rows


DETAIL_FIELDS = [
    "source_file",
    "model_name",
    "example_id",
    "domain",
    "privacy_level",
    "attack_strength",
    "privacy_score",
    "utility_score",
    "example_score",
    "leaked_count",
    "allowed_count",
    "leaked_items",
    "allowed_items",
    "do_not_disclose_values",
    "allowed_values",
    "transcript_text",
    "transcript_json",
]


# =========================================================
# Main
# =========================================================
def main() -> None:
    json_files = sorted(INPUT_DIR.glob("*.json"))
    if not json_files:
        print(f"No JSON files found in {INPUT_DIR}")
        return

    all_summary_rows: List[Dict[str, Any]] = []
    all_detail_rows: List[Dict[str, Any]] = []

    for json_path in json_files:
        try:
            with json_path.open("r", encoding="utf-8") as f:
                data = json.load(f)

            if is_summary_json(data):
                rows = flatten_summary_json(data, source_file=json_path.name)
                if rows:
                    out_path = OUTPUT_DIR / f"{json_path.stem}.csv"
                    write_csv(rows, out_path, SUMMARY_FIELDS)
                    all_summary_rows.extend(rows)
                    print(f"Converted summary: {json_path} -> {out_path}")
                else:
                    print(f"Skipped summary {json_path.name}: no rows parsed")

            elif is_details_json(data):
                rows = flatten_details_json(data, source_file=json_path.name)
                if rows:
                    out_path = OUTPUT_DIR / f"{json_path.stem}.csv"
                    write_csv(rows, out_path, DETAIL_FIELDS)
                    all_detail_rows.extend(rows)
                    print(f"Converted details: {json_path} -> {out_path}")
                else:
                    print(f"Skipped details {json_path.name}: no rows parsed")

            else:
                print(f"Skipped {json_path.name}: unrecognized JSON structure")

        except Exception as e:
            print(f"Failed to convert {json_path}: {e}")

    if all_summary_rows:
        combined_summary_path = OUTPUT_DIR / "all_summary_results.csv"
        write_csv(all_summary_rows, combined_summary_path, SUMMARY_FIELDS)
        print(f"Combined summary CSV written to: {combined_summary_path}")

    if all_detail_rows:
        combined_detail_path = OUTPUT_DIR / "all_detail_results.csv"
        write_csv(all_detail_rows, combined_detail_path, DETAIL_FIELDS)
        print(f"Combined details CSV written to: {combined_detail_path}")


if __name__ == "__main__":
    main()

Converted details: data\leaderboard\details_Apertus-8B-Instruct-2509.json -> data\leaderboard_csv\details_Apertus-8B-Instruct-2509.csv
Converted details: data\leaderboard\GLM-4.7-Flash.json -> data\leaderboard_csv\GLM-4.7-Flash.csv
Skipped GLM-4.7-Flash.json.checkpoint.json: unrecognized JSON structure
Converted summary: data\leaderboard\summary_Apertus-8B-Instruct-2509.json -> data\leaderboard_csv\summary_Apertus-8B-Instruct-2509.csv
Combined summary CSV written to: data\leaderboard_csv\all_summary_results.csv
Combined details CSV written to: data\leaderboard_csv\all_detail_results.csv
